<i>Copyright (c) Recommenders contributors.</i>

<i>Licensed under the MIT License.</i>

# DKN : Deep Knowledge-Aware Network for News Recommendation

DKN \[1\] is a deep learning model which incorporates information from knowledge graph for better news recommendation. Specifically, DKN uses TransX \[2\] method for knowledge graph representation learning, then applies a CNN framework, named KCNN, to combine entity embedding with word embedding and generate a final embedding vector for a news article. CTR prediction is made via an attention-based neural scorer. 

## Properties of DKN:

- DKN is a content-based deep model for CTR prediction rather than traditional ID-based collaborative filtering. 
- It makes use of knowledge entities and common sense in news content via joint learning from semantic-level and knowledge-level representations of news articles.
- DKN uses an attention module to dynamically calculate a user's aggregated historical representation.


## Data format

DKN takes several files as input as follows:

- **training / validation / test files**: each line in these files represents one instance. Impressionid is used to evaluate performance within an impression session, so it is only used when evaluating, you can set it to 0 for training data. The format is : <br> 
`[label] [userid] [CandidateNews]%[impressionid] `<br> 
e.g., `1 train_U1 N1%0` <br> 

- **user history file**: each line in this file represents a users' click history. The `history_size` argument of the model is the max number of user's click history we use. We will automatically keep the last `history_size` number of user click history, if user's click history is more than `history_size`, and we will automatically pad with 0 if user's click history is less than `history_size`. the format is : <br> 
`[Userid] [newsid1,newsid2...]`<br>
e.g., `train_U1 N1,N2` <br> 

- **document feature file**: It contains the word and entity features for news articles. News articles are represented by aligned title words and title entities. To take a quick example, a news title may be: <i>"Trump to deliver State of the Union address next week"</i>, then the title words value may be `CandidateNews:34,45,334,23,12,987,3456,111,456,432` and the title entitie value may be: `entity:45,0,0,0,0,0,0,0,0,0`. Only the first value of entity vector is non-zero due to the word "Trump". The title value and entity value is hashed from 1 to `n` (where `n` is the number of distinct words or entities). Each feature length should be fixed at k, if the number of words in document is more than k, you should truncate the document to k words, and if the number of words in document is less than k, you should pad 0 to the end. 
the format is like: <br> 
`[Newsid] [w1,w2,w3...wk] [e1,e2,e3...ek]`

- **word embedding/entity embedding/ context embedding files**: These are `*.npy` files of pretrained embeddings. After loading, each file is a `[n+1,k]` two-dimensional matrix, n is the number of words(or entities) of their hash dictionary, k is dimension of the embedding, note that we keep embedding 0 for zero padding. 

In this experiment, we used GloVe\[4\] vectors to initialize the word embedding. We trained entity embedding using TransE\[2\] on knowledge graph and context embedding is the average of the entity's neighbors in the knowledge graph.<br>

## MIND dataset

MIND dataset\[3\] is a large-scale English news dataset. It was collected from anonymized behavior logs of Microsoft News website. MIND contains 1,000,000 users, 161,013 news articles and 15,777,377 impression logs. Every news article contains rich textual content including title, abstract, body, category and entities. Each impression log contains the click events, non-clicked events and historical news click behaviors of this user before this impression.

A smaller version, [MIND-small](https://azure.microsoft.com/en-us/services/open-datasets/catalog/microsoft-news-dataset/), is a small version of the MIND dataset by randomly sampling 50,000 users and their behavior logs from the MIND dataset.

The datasets contains these files for both training and validation data:

#### behaviors.tsv

The behaviors.tsv file contains the impression logs and users' news click hostories. It has 5 columns divided by the tab symbol:

+ Impression ID. The ID of an impression.
+ User ID. The anonymous ID of a user.
+ Time. The impression time with format "MM/DD/YYYY HH:MM:SS AM/PM".
+ History. The news click history (ID list of clicked news) of this user before this impression.
+ Impressions. List of news displayed in this impression and user's click behaviors on them (1 for click and 0 for non-click).

One simple example: 

`1    U82271    11/11/2019 3:28:58 PM    N3130 N11621 N12917 N4574 N12140 N9748    N13390-0 N7180-0 N20785-0 N6937-0 N15776-0 N25810-0 N20820-0 N6885-0 N27294-0 N18835-0 N16945-0 N7410-0 N23967-0 N22679-0 N20532-0 N26651-0 N22078-0 N4098-0 N16473-0 N13841-0 N15660-0 N25787-0 N2315-0 N1615-0 N9087-0 N23880-0 N3600-0 N24479-0 N22882-0 N26308-0 N13594-0 N2220-0 N28356-0 N17083-0 N21415-0 N18671-0 N9440-0 N17759-0 N10861-0 N21830-0 N8064-0 N5675-0 N15037-0 N26154-0 N15368-1 N481-0 N3256-0 N20663-0 N23940-0 N7654-0 N10729-0 N7090-0 N23596-0 N15901-0 N16348-0 N13645-0 N8124-0 N20094-0 N27774-0 N23011-0 N14832-0 N15971-0 N27729-0 N2167-0 N11186-0 N18390-0 N21328-0 N10992-0 N20122-0 N1958-0 N2004-0 N26156-0 N17632-0 N26146-0 N17322-0 N18403-0 N17397-0 N18215-0 N14475-0 N9781-0 N17958-0 N3370-0 N1127-0 N15525-0 N12657-0 N10537-0 N18224-0 `

#### news.tsv

The news.tsv file contains the detailed information of news articles involved in the behaviors.tsv file. It has 7 columns, which are divided by the tab symbol:

+ News ID
+ Category
+ SubCategory
+ Title
+ Abstract
+ URL
+ Title Entities (entities contained in the title of this news)
+ Abstract Entities (entites contained in the abstract of this news)

One simple example: 

`N46466    lifestyle    lifestyleroyals    The Brands Queen Elizabeth, Prince Charles, and Prince Philip Swear By    Shop the notebooks, jackets, and more that the royals can't live without.    https://www.msn.com/en-us/lifestyle/lifestyleroyals/the-brands-queen-elizabeth,-prince-charles,-and-prince-philip-swear-by/ss-AAGH0ET?ocid=chopendata    [{"Label": "Prince Philip, Duke of Edinburgh", "Type": "P", "WikidataId": "Q80976", "Confidence": 1.0, "OccurrenceOffsets": [48], "SurfaceForms": ["Prince Philip"]}, {"Label": "Charles, Prince of Wales", "Type": "P", "WikidataId": "Q43274", "Confidence": 1.0, "OccurrenceOffsets": [28], "SurfaceForms": ["Prince Charles"]}, {"Label": "Elizabeth II", "Type": "P", "WikidataId": "Q9682", "Confidence": 0.97, "OccurrenceOffsets": [11], "SurfaceForms": ["Queen Elizabeth"]}]    [] `

#### entity_embedding.vec & relation_embedding.vec

The entity_embedding.vec and relation_embedding.vec files contain the 100-dimensional embeddings of the entities and relations learned from the subgraph (from WikiData knowledge graph) by TransE method. In both files, the first column is the ID of entity/relation, and the other columns are the embedding vector values.

One simple example: 

`Q42306013  0.014516 -0.106958 0.024590 ... -0.080382`


## DKN architecture

The following figure shows the architecture of DKN.

![](https://raw.githubusercontent.com/recommenders-team/resources/main/images/dkn_architecture.png)

DKN takes one piece of candidate news and one piece of a user’s clicked news as input. For each piece of news, a specially designed KCNN is used to process its title and generate an embedding vector. KCNN is an extension of traditional CNN that allows flexibility in incorporating symbolic knowledge from a knowledge graph into sentence representation learning. 

With the KCNN, we obtain a set of embedding vectors for a user’s clicked history. To get final embedding of the user with
respect to the current candidate news, we use an attention-based method to automatically match the candidate news to each piece
of his clicked news, and aggregate the user’s historical interests with different weights. The candidate news embedding and the user embedding are concatenated and fed into a deep neural network (DNN) to calculate the predicted probability that the user will click the candidate news.

## Global settings and imports

In [1]:
import os
import sys
from tempfile import TemporaryDirectory

import torch

from recommenders.datasets.mind import (
    download_mind,
    extract_mind,
    read_clickhistory,
    get_train_input,
    get_valid_input,
    get_user_history,
    get_words_and_entities,
    generate_embeddings,
)
from recommenders.models.deeprec.models.pytorch.dkn import DKN
from recommenders.utils.notebook_utils import store_metadata

print(f"System version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")

System version: 3.11.14 (main, Jan 14 2026, 19:35:32) [Clang 21.1.4 ]
PyTorch version: 2.13.0.dev20260521+cu132


In [2]:
# Temp dir
tmpdir = TemporaryDirectory()


In [3]:
# Mind parameters
MIND_SIZE = "small"

# DKN parameters
EPOCHS = 10
HISTORY_SIZE = 50
BATCH_SIZE = 100
RANDOM_SEED = 42  # Set this to None for non-deterministic result

# Paths
data_path = os.path.join(tmpdir.name, "mind-dkn")
train_file = os.path.join(data_path, "train_mind.txt")
valid_file = os.path.join(data_path, "valid_mind.txt")
user_history_file = os.path.join(data_path, "user_history.txt")
infer_embedding_file = os.path.join(data_path, "infer_embedding.txt")

## Data preparation

In this example, let's go through a real case on how to apply DKN on a raw news dataset from the very beginning. We will download a copy of open-source MIND dataset, in its original raw format. Then we will process the raw data files into DKN's input data format, which is stated previously. 

In [4]:
train_zip, valid_zip = download_mind(size=MIND_SIZE, dest_path=data_path)
train_path, valid_path = extract_mind(train_zip, valid_zip)

  0%|          | 0.00/51.8k [00:00<?, ?KB/s]

  0%|          | 1.00/51.8k [00:01<16:11:59, 1.13s/KB]

  0%|          | 149/51.8k [00:01<05:08, 167KB/s]     

  3%|▎         | 1.52k/51.8k [00:01<00:24, 2.04kKB/s]

 14%|█▎        | 7.09k/51.8k [00:01<00:04, 10.9kKB/s]

...

 16%|█▋        | 4.95k/30.2k [00:01<00:02, 9.93kKB/s]

 37%|███▋      | 11.1k/30.2k [00:01<00:00, 20.0kKB/s]

 55%|█████▌    | 16.7k/30.2k [00:01<00:00, 28.2kKB/s]

 77%|███████▋  | 23.2k/30.2k [00:01<00:00, 37.4kKB/s]

100%|██████████| 30.2k/30.2k [00:01<00:00, 20.8kKB/s]

In [5]:
train_session, train_history = read_clickhistory(train_path, "behaviors.tsv")
valid_session, valid_history = read_clickhistory(valid_path, "behaviors.tsv")
get_train_input(train_session, train_file)
get_valid_input(valid_session, valid_file)
get_user_history(train_history, valid_history, user_history_file)

In [6]:
train_news = os.path.join(train_path, "news.tsv")
valid_news = os.path.join(valid_path, "news.tsv")
news_words, news_entities = get_words_and_entities(train_news, valid_news)

In [7]:
train_entities = os.path.join(train_path, "entity_embedding.vec")
valid_entities = os.path.join(valid_path, "entity_embedding.vec")
news_feature_file, word_embeddings_file, entity_embeddings_file = generate_embeddings(
    data_path,
    news_words,
    news_entities,
    train_entities,
    valid_entities,
    max_sentence=10,
    word_embedding_dim=100,
)

  0%|          | 0.00/842k [00:00<?, ?KB/s]

  0%|          | 1.00/842k [00:00<77:18:36, 3.03KB/s]

  0%|          | 55.0/842k [00:00<1:46:18, 132KB/s]  

  0%|          | 140/842k [00:00<52:05, 269KB/s]   

  0%|          | 300/842k [00:00<27:29, 510KB/s]

...

 97%|█████████▋| 821k/842k [00:24<00:00, 64.7kKB/s]

 98%|█████████▊| 827k/842k [00:24<00:00, 60.6kKB/s]

 99%|█████████▉| 833k/842k [00:24<00:00, 59.3kKB/s]

100%|█████████▉| 840k/842k [00:24<00:00, 61.1kKB/s]

100%|██████████| 842k/842k [00:24<00:00, 34.1kKB/s]

## Create the DKN model

The architecture is set on the constructor and the training knobs on `fit`. `generate_embeddings` produces word and entity embeddings but no context embeddings, so the model uses the word and entity channels only. Its convolution, attention and scoring layers are initialized with a Glorot normal initializer.

In [8]:
model = DKN(
    news_feature_file=news_feature_file,
    user_history_file=user_history_file,
    word_embedding_file=word_embeddings_file,
    entity_embedding_file=entity_embeddings_file,
    history_size=HISTORY_SIZE,
    filter_sizes=[1, 2, 3],
    num_filters=100,
    attention_layer_size=100,
    layer_sizes=[300],
    enable_BN=True,
    init_method="xavier_normal",
    seed=RANDOM_SEED,
)

## Train the DKN model

In [9]:
%%time
model = model.fit(
    train_file,
    valid_file,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=0.0003,
    embed_l2=0.0,
    layer_l2=1e-6,
)

step 10000 , total_loss: 0.4832, data_loss: 0.4829


at epoch 1
train info: loss:0.46646614952486043
eval info: auc:0.6641, group_auc:0.6213, mean_mrr:0.2695, ndcg@10:0.3618, ndcg@5:0.2914


step 10000 , total_loss: 0.4757, data_loss: 0.4753


at epoch 2
train info: loss:0.4537634518590751
eval info: auc:0.6678, group_auc:0.6304, mean_mrr:0.2772, ndcg@10:0.3689, ndcg@5:0.2989


step 10000 , total_loss: 0.4635, data_loss: 0.4630


...

at epoch 8
train info: loss:0.4226322054855411
eval info: auc:0.6464, group_auc:0.6277, mean_mrr:0.28, ndcg@10:0.369, ndcg@5:0.3041


step 10000 , total_loss: 0.4513, data_loss: 0.4503


at epoch 9
train info: loss:0.41718326292378505
eval info: auc:0.6411, group_auc:0.6226, mean_mrr:0.2783, ndcg@10:0.3665, ndcg@5:0.3012


step 10000 , total_loss: 0.4432, data_loss: 0.4420


at epoch 10
train info: loss:0.41129259535162394
eval info: auc:0.6354, group_auc:0.6176, mean_mrr:0.2756, ndcg@10:0.3629, ndcg@5:0.2971
CPU times: user 1h 48min 33s, sys: 12min 48s, total: 2h 1min 21s
Wall time: 2h 1min 40s


## Evaluate the DKN model

In [10]:
res = model.run_eval(valid_file, batch_size=BATCH_SIZE)
print(res)

{'auc': 0.6354, 'group_auc': 0.6176, 'mean_mrr': 0.2756, 'ndcg@5': 0.2971, 'ndcg@10': 0.3629}


In [11]:
# Record results for tests - ignore this cell
store_metadata("auc", res["auc"])
store_metadata("group_auc", res["group_auc"])
store_metadata("ndcg@5", res["ndcg@5"])
store_metadata("ndcg@10", res["ndcg@10"])
store_metadata("mean_mrr", res["mean_mrr"])

## Document embedding inference API

After training, you can get document embedding through this document embedding inference API. The input file format is same with document feature file. The output file fomrat is: `[Newsid] [embedding]`

In [12]:
model = model.run_get_embedding(news_feature_file, infer_embedding_file, batch_size=BATCH_SIZE)

In [13]:
# Cleanup
tmpdir.cleanup()

## Results on large MIND dataset

Here are performances using the large MIND dataset (1,000,000 users, 161,013 news articles and 15,777,377 impression logs). 

| Models | g-AUC | MRR |NDCG@5 | NDCG@10 |
| :------| :------: | :------: | :------: | :------ |
| LibFM | 0.5993 | 0.2823 | 0.3005 | 0.3574 |
| Wide&Deep | 0.6216 | 0.2931 | 0.3138 | 0.3712 |
| DKN | 0.6436 | 0.3128 | 0.3371 | 0.3908|


Note that the results of DKN were obtained with the TensorFlow implementation that this PyTorch model replaces, and the results of the first two models come from the MIND paper \[3\].
We compare the results on the same test dataset. 

One epoch takes 6381.3s (5066.6s for training, 1314.7s for evaluating) for DKN on GPU. Hardware specification for running the large dataset: <br>
GPU: Tesla P100-PCIE-16GB <br>
CPU: 6 cores Intel(R) Xeon(R) CPU E5-2690 v4 @ 2.60GHz

## References

\[1\] Hongwei Wang, Fuzheng Zhang, Xing Xie and Minyi Guo, "DKN: Deep Knowledge-Aware Network for News Recommendation", in Proceedings of the 2018 World Wide Web Conference (WWW), 2018, https://arxiv.org/abs/1801.08284. <br>
\[2\] Knowledge Graph Embeddings including TransE, TransH, TransR and PTransE. https://github.com/thunlp/KB2E <br>
\[3\] Fangzhao Wu et al., "MIND: A Large-scale Dataset for News Recommendation", Proceedings of the 58th Annual Meeting of the Association for Computational Linguistics, 2020, https://msnews.github.io/competition.html. <br>
\[4\] GloVe: Global Vectors for Word Representation. https://nlp.stanford.edu/projects/glove/